# Title

In [2]:
print("hello world")

hello world


# Task:
1. Download CSV files from my repo
2. Load them into notebook using Pandas
3. Clean the CSVs.
4. Output clean files

In [3]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [32]:
# Locate the CSV files
from pathlib import Path
print("Notebook folder:", Path.cwd())
csv_files = list(Path.cwd().rglob("*.csv"))
for file in csv_files:
    print(file)

Notebook folder: c:\Users\Admin\Desktop\20260817-DE5M5
c:\Users\Admin\Desktop\20260817-DE5M5\03_Library Systembook.csv
c:\Users\Admin\Desktop\20260817-DE5M5\03_Library SystemCustomers.csv
c:\Users\Admin\Desktop\20260817-DE5M5\clean_data\library_books_clean.csv
c:\Users\Admin\Desktop\20260817-DE5M5\clean_data\library_customers_clean.csv


In [8]:
# Load both files
import pandas as pd
books_path = next(
        file for file in csv_files
        if "Systembook" in file.name
)
customers_path = next(
        file for file in csv_files
        if "SystemCustomer" in file.name
)
books_df = pd.read_csv(books_path)
customers_df = pd.read_csv(customers_path)

In [9]:
# Preview the data
display(books_df.head())
display(customers_df.head())

,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID
0,1.0,Catcher in the Rye,"""20/02/2023""",25/02/2023,2 weeks,1.0
1,2.0,Lord of the rings the two towers,"""24/03/2023""",21/03/2023,2 weeks,2.0
2,3.0,Lord of the rings the return of the kind,"""29/03/2023""",25/03/2023,2 weeks,3.0
3,4.0,The hobbit,"""02/04/2023""",25/03/2023,2 weeks,4.0
4,5.0,Dune,"""02/04/2023""",25/03/2023,2 weeks,5.0


,Customer ID,Customer Name
0,1.0,Jane Doe
1,2.0,John Smith
2,3.0,Dan Reeves
3,NaN,NaN
4,5.0,William Holden


In [10]:
# Check their structure
print("Books shape:", books_df.shape)
print("Customers shape:", customers_df.shape)

books_df.info()
customers_df.info()

Books shape: (114, 6)
Customers shape: (9, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Id                      21 non-null     float64
 1   Books                   20 non-null     object 
 2   Book checkout           21 non-null     object 
 3   Book Returned           21 non-null     object 
 4   Days allowed to borrow  21 non-null     object 
 5   Customer ID             20 non-null     float64
dtypes: float64(2), object(4)
memory usage: 5.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Customer ID    8 non-null      float64
 1   Customer Name  8 non-null      object 
dtypes: float64(1), object(1)
memory usage: 276.0+ bytes


In [13]:
# Remove empty rows
books_clean = books_df.copy()
customers_clean = customers_df.copy()

books_clean = books_clean.dropna(how="all")
customers_clean = customers_clean.dropna(how="all")

# convert names to lowercase
books_clean.columns = (
    books_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)
customers_clean.columns = (
    customers_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

print("Books:", books_clean.shape)
print("Customers:", customers_clean.shape)

Books: (21, 6)
Customers: (8, 2)


In [17]:
# Clean text values
text_columns = ["books", "book_checkout", "book_returned", "days_allowed_to_borrow"] 

for column in text_columns:
    books_clean[column] = (
        books_clean[column]
        .astype("string")
        .str.strip()
        .str.strip('"')
    )
    customers_clean["customer_name"] = (
        customers_clean["customer_name"]
        .astype("string")
        .str.strip()
    )    

In [18]:
# Dates use day/month/year
books_clean["book_checkout"] = pd.to_datetime(
    books_clean["book_checkout"],
    format="%d/%m/%y",
    errors="coerce"
)
books_clean["book_returned"] = pd.to_datetime(
    books_clean["book_returned"],
    format="%d/%m/%y",
    errors="coerce"
)

In [19]:
# Remove duplicates
books_clean = books_clean.drop_duplicates()
customers_clean = customers_clean.drop_duplicates()

# Unique customer ID
customers_clean = customers_clean.drop_duplicates(
    subset="customer_id",
    keep="first"
)

In [20]:
# Missing values
print("Missing book values:")
display(books_clean.isna().sum())

print("Missing customers values:")
display(customers_clean.isna().sum())

Missing book values:


id                         0
books                      1
book_checkout             21
book_returned             21
days_allowed_to_borrow     0
customer_id                1
dtype: int64

Missing customers values:


customer_id      0
customer_name    0
dtype: int64

In [23]:
# IDs: convert values such as 1.0 to integer 1
books_clean["id"] = pd.to_numeric(
    books_clean["id"], errors="coerce"
).astype("Int64")

books_clean["customer_id"] = pd.to_numeric(
    books_clean["customer_id"], errors="coerce"
).astype("Int64")

customers_clean["customer_id"] = pd.to_numeric(
    customers_clean["customer_id"], errors="coerce"
).astype("Int64")

In [24]:
#Convert “2 weeks” into days
weeks = books_clean["days_allowed_to_borrow"].str.extract(r"(\d+)")[0]


books_clean["days_allowed"] = (
    pd.to_numeric(weeks, errors="coerce") * 7
).astype("Int64")


books_clean = books_clean.drop(columns="days_allowed_to_borrow")

In [26]:
# Incorrect dates
books_clean["loan_days"] = (
    books_clean["book_returned"] -
    books_clean["book_checkout"]
).dt.days

invalid_dates = books_clean[
    books_clean["loan_days"] < 0
]

display(invalid_dates)

,id,books,book_checkout,book_returned,customer_id,loan_days,days_allowed


In [27]:
# Books returned late
books_clean["returned_late"] = (
    books_clean["loan_days"] >
    books_clean["days_allowed"]
)

In [29]:
# Customer relationships
unknown_customers = books_clean[
    books_clean["customer_id"].notna()
    & ~books_clean["customer_id"].isin(customers_clean["customer_id"])
    ]

display(unknown_customers)

,id,books,book_checkout,book_returned,customer_id,loan_days,days_allowed,returned_late
3,4,The hobbit,NaT,NaT,4,NaN,14,<NA>
18,19,Dracula,NaT,NaT,10,NaN,14,<NA>


In [30]:
# Review data
display(books_clean.head())
display(customers_clean.head())

books_clean.info()
customers_clean.info()

,id,books,book_checkout,book_returned,customer_id,loan_days,days_allowed,returned_late
0,1,Catcher in the Rye,NaT,NaT,1,NaN,14,<NA>
1,2,Lord of the rings the two towers,NaT,NaT,2,NaN,14,<NA>
2,3,Lord of the rings the return of the kind,NaT,NaT,3,NaN,14,<NA>
3,4,The hobbit,NaT,NaT,4,NaN,14,<NA>
4,5,Dune,NaT,NaT,5,NaN,14,<NA>


,customer_id,customer_name
0,1,Jane Doe
1,2,John Smith
2,3,Dan Reeves
4,5,William Holden
5,6,Jaztyn Forest


<class 'pandas.core.frame.DataFrame'>
Index: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id             21 non-null     Int64         
 1   books          20 non-null     string        
 2   book_checkout  0 non-null      datetime64[ns]
 3   book_returned  0 non-null      datetime64[ns]
 4   customer_id    20 non-null     Int64         
 5   loan_days      0 non-null      float64       
 6   days_allowed   21 non-null     Int64         
 7   returned_late  0 non-null      boolean       
dtypes: Int64(3), boolean(1), datetime64[ns](2), float64(1), string(1)
memory usage: 1.4 KB
<class 'pandas.core.frame.DataFrame'>
Index: 8 entries, 0 to 8
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_id    8 non-null      Int64 
 1   customer_name  8 non-null      string
dtypes: Int64(1), string(1)
memory

In [31]:
from pathlib import Path

output_folder = Path("clean_data")
output_folder.mkdir(exist_ok=True)

books_clean.to_csv(
    output_folder / "library_books_clean.csv",
    index=False,
    date_format="%Y-%m-%d"
)

customers_clean.to_csv(
    output_folder / "library_customers_clean.csv",
    index=False
)

# Verify agan the output files

In [33]:
#Reload original files
books_df = pd.read_csv(books_path)

books_clean = books_df.dropna(how="all").copy()

books_clean.columns = (
    books_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

In [34]:
# Remove quotes before converting the dates
date_columns = [
    "book_checkout",
    "book_returned"
]

for date_column in date_columns:
    books_clean[date_column] = (
        books_clean[date_column]
        .astype("string")
        .str.replace('"', "", regex=False)
        .str.strip()
    )

In [35]:
# Check the cleaned dates
display(
    books_clean[
        ["book_checkout", "book_returned"]
    ].head()
)

,book_checkout,book_returned
0,20/02/2023,25/02/2023
1,24/03/2023,21/03/2023
2,29/03/2023,25/03/2023
3,02/04/2023,25/03/2023
4,02/04/2023,25/03/2023


In [36]:
# Convert to real dates
for date_column in date_columns:
    original_values = books_clean[date_column].copy()

    parsed_values = pd.to_datetime(
        original_values,
        format="%d/%m/%Y",
        errors="coerce"
    )

    invalid_mask = (
        original_values.notna()
        & parsed_values.isna()
    )

    if invalid_mask.any():
        print(f"Invalid values in {date_column}:")
        display(
            books_clean.loc[
                invalid_mask,
                ["id", "books", date_column]
            ]
        )

    books_clean[date_column] = parsed_values

Invalid values in book_checkout:


,id,books,book_checkout
16,17.0,The Bloody Chamber,32/05/2023


In [38]:
# Convert values such as "2 weeks" into 14 days
number_of_weeks = (
    books_clean["days_allowed_to_borrow"]
    .astype("string")
    .str.extract(r"(\d+)")[0]
)

books_clean["days_allowed"] = (
    pd.to_numeric(number_of_weeks, errors="coerce") * 7
).astype("Int64")

In [39]:
# Recalculate the loan duration
books_clean["loan_days"] = (
    books_clean["book_returned"]
    - books_clean["book_checkout"]
).dt.days


In [40]:
# Validation columns
books_clean["invalid_date_order"] = (
    books_clean["loan_days"] < 0
)

books_clean["returned_late"] = (
    books_clean["loan_days"]
    > books_clean["days_allowed"]
)

In [41]:
# Verify till now
display(
    books_clean[
        [
            "books",
            "book_checkout",
            "book_returned",
            "loan_days",
            "days_allowed",
            "returned_late",
            "invalid_date_order"
        ]
    ]
)

,books,book_checkout,book_returned,loan_days,days_allowed,returned_late,invalid_date_order
0,Catcher in the Rye,2023-02-20,2023-02-25,5.0,14,False,False
1,Lord of the rings the two towers,2023-03-24,2023-03-21,-3.0,14,False,True
2,Lord of the rings the return of the kind,2023-03-29,2023-03-25,-4.0,14,False,True
3,The hobbit,2023-04-02,2023-03-25,-8.0,14,False,True
4,Dune,2023-04-02,2023-03-25,-8.0,14,False,True
5,Little Women,2023-04-02,2023-05-01,29.0,14,True,False
6,IT,2063-04-10,2023-04-03,-14617.0,14,False,True
7,Misery,2023-04-15,2023-04-03,-12.0,14,False,True
8,Catch 22,2023-04-15,2023-04-16,1.0,14,False,False
9,Animal Farm,2023-04-20,2023-04-24,4.0,14,False,False


In [ ]:
# Current Columns
print(books_clean.columns.tolist())

['id', 'books', 'book_checkout', 'book_returned', 'days_allowed_to_borrow', 'customer_id', 'loan_days', 'days_allowed', 'invalid_date_order', 'returned_late']


In [43]:
# A loan duration is valid only when both dates exist
# and the return is not before checkout.
valid_duration = (
    books_clean["book_checkout"].notna()
    & books_clean["book_returned"].notna()
    & books_clean["loan_days"].ge(0)
    & books_clean["days_allowed"].notna()
)

# Use a nullable Boolean column
books_clean["returned_late"] = pd.Series(
    pd.NA,
    index=books_clean.index,
    dtype="boolean"
)

books_clean.loc[valid_duration, "returned_late"] = (
    books_clean.loc[valid_duration, "loan_days"]
    > books_clean.loc[valid_duration, "days_allowed"]
)

In [44]:
#Records with issues
required_columns = [
    "id",
    "books",
    "book_checkout",
    "book_returned",
    "customer_id"
]

missing_required_value = (
    books_clean[required_columns]
    .isna()
    .any(axis=1)
)

invalid_date_order = (
    books_clean["loan_days"] < 0
)

suspicious_year = (
    books_clean["book_checkout"].dt.year > 2026
) | (
    books_clean["book_returned"].dt.year > 2026
)

issue_mask = (
    missing_required_value
    | invalid_date_order
    | suspicious_year
)

In [45]:
# Add an explanation for each issue
def identify_issue(row):
    issues = []

    if pd.isna(row["books"]):
        issues.append("missing book title")

    if pd.isna(row["customer_id"]):
        issues.append("missing customer ID")

    if pd.isna(row["book_checkout"]):
        issues.append("missing or invalid checkout date")

    if pd.isna(row["book_returned"]):
        issues.append("missing or invalid return date")

    if pd.notna(row["loan_days"]) and row["loan_days"] < 0:
        issues.append("return date precedes checkout date")

    checkout_year = (
        row["book_checkout"].year
        if pd.notna(row["book_checkout"])
        else None
    )

    if checkout_year and checkout_year > 2026:
        issues.append("suspicious checkout year")

    return "; ".join(issues)

In [48]:
# Create both clean and issue DB
books_issues = books_clean.loc[issue_mask].copy()
books_issues["data_quality_issue"] = books_issues.apply(
    identify_issue,
    axis=1
)

books_final = books_clean.loc[~issue_mask].copy()

In [49]:
# Review the outputs
print("Valid books:", len(books_final))
print("Books requiring review:", len(books_issues))

display(books_final)
display(books_issues)

Valid books: 13
Books requiring review: 8


,id,books,book_checkout,book_returned,days_allowed_to_borrow,customer_id,loan_days,days_allowed,invalid_date_order,returned_late
0,1.0,Catcher in the Rye,2023-02-20,2023-02-25,2 weeks,1.0,5.0,14,False,False
5,6.0,Little Women,2023-04-02,2023-05-01,2 weeks,1.0,29.0,14,False,True
8,9.0,Catch 22,2023-04-15,2023-04-16,2 weeks,7.0,1.0,14,False,False
9,10.0,Animal Farm,2023-04-20,2023-04-24,2 weeks,2.0,4.0,14,False,False
10,11.0,1984,2023-04-23,2023-04-27,2 weeks,8.0,4.0,14,False,False
11,12.0,Little Women,2023-04-02,2023-05-01,2 weeks,1.0,29.0,14,False,True
12,13.0,East of Eden,2023-04-30,2023-05-05,2 weeks,2.0,5.0,14,False,False
13,14.0,America Is in the Heart,2023-05-01,2023-05-07,2 weeks,3.0,6.0,14,False,False
14,15.0,Wuthering Heights,2023-05-01,2023-05-10,2 weeks,9.0,9.0,14,False,False
15,16.0,Dark Tales,2023-05-15,2023-06-01,2 weeks,2.0,17.0,14,False,True


,id,books,book_checkout,book_returned,days_allowed_to_borrow,customer_id,loan_days,days_allowed,invalid_date_order,returned_late,data_quality_issue
1,2.0,Lord of the rings the two towers,2023-03-24,2023-03-21,2 weeks,2.0,-3.0,14,True,<NA>,return date precedes checkout date
2,3.0,Lord of the rings the return of the kind,2023-03-29,2023-03-25,2 weeks,3.0,-4.0,14,True,<NA>,return date precedes checkout date
3,4.0,The hobbit,2023-04-02,2023-03-25,2 weeks,4.0,-8.0,14,True,<NA>,return date precedes checkout date
4,5.0,Dune,2023-04-02,2023-03-25,2 weeks,5.0,-8.0,14,True,<NA>,return date precedes checkout date
6,7.0,IT,2063-04-10,2023-04-03,2 weeks,6.0,-14617.0,14,True,<NA>,return date precedes checkout date; suspicious...
7,8.0,Misery,2023-04-15,2023-04-03,2 weeks,7.0,-12.0,14,True,<NA>,return date precedes checkout date
16,17.0,The Bloody Chamber,NaT,2023-06-04,2 weeks,3.0,NaN,14,False,<NA>,missing or invalid checkout date
20,21.0,NaN,2023-06-01,2023-06-05,2 weeks,NaN,4.0,14,False,False,missing book title; missing customer ID


In [50]:
# Clean Customer table
customer_issue_mask = (
    customers_clean[["customer_id", "customer_name"]]
    .isna()
    .any(axis=1)
    | customers_clean.duplicated(
        subset="customer_id",
        keep=False
    )
)

customers_issues = customers_clean.loc[
    customer_issue_mask
].copy()

customers_final = customers_clean.loc[
    ~customer_issue_mask
].copy()

In [51]:
#Book with valid customer
known_customer_ids = set(
    customers_final["customer_id"].dropna()
)

unknown_customer_mask = (
    ~books_final["customer_id"].isin(known_customer_ids)
)

display(
    books_final.loc[
        unknown_customer_mask,
        ["id", "books", "customer_id"]
    ]
)

,id,books,customer_id
18,19.0,Dracula,10.0


In [52]:
# Save outputs
from pathlib import Path

output_folder = Path("clean_data")
output_folder.mkdir(exist_ok=True)

books_final.to_csv(
    output_folder / "library_books_clean.csv",
    index=False,
    date_format="%Y-%m-%d"
)

customers_final.to_csv(
    output_folder / "library_customers_clean.csv",
    index=False
)

books_issues.to_csv(
    output_folder / "library_books_issues.csv",
    index=False,
    date_format="%Y-%m-%d"
)

customers_issues.to_csv(
    output_folder / "library_customers_issues.csv",
    index=False
)

## Function to calculate the difference between the date columns
## Try using SQLAlchemy to write your clean files to SSMS